In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import os
from hflayers import HopfieldLayer

# ==========================================
# 1. CONFIGURATION & DATA PREPARATION
# ==========================================
data_dir = "simulation_data"
num_cells = 20
total_frames = 120
features_per_frame = num_cells * 5  # 100
total_input_size = total_frames * features_per_frame  # 12000 dimensions

print("Extracting FULL 120 frames for Backpropagation Training...")

X_train = []
Y_train = []

for i in range(35):
    # Killing data -> Label [1.0, 0.0]
    k_data = np.load(os.path.join(data_dir, f"data_killing_{i}.npy"))
    k_full = k_data.transpose(1, 0, 2).flatten() 
    X_train.append(torch.tensor(k_full, dtype=torch.float32))
    Y_train.append(torch.tensor([1.0, 0.0], dtype=torch.float32))
    
    # Non-Killing data -> Label [0.0, 1.0]
    nk_data = np.load(os.path.join(data_dir, f"data_non-killing_{i}.npy"))
    nk_full = nk_data.transpose(1, 0, 2).flatten()
    X_train.append(torch.tensor(nk_full, dtype=torch.float32))
    Y_train.append(torch.tensor([0.0, 1.0], dtype=torch.float32))

X_train = torch.stack(X_train).unsqueeze(1) # Shape: [70, 1, 12000]
Y_train = torch.stack(Y_train)



Extracting FULL 120 frames for Backpropagation Training...


In [ ]:
# ==========================================
# 2. CHNN LAYER WITH FULL TRAINABLE INPUTS
# ==========================================
chnn_classifier = HopfieldLayer(
    input_size=total_input_size,    # 12000 dimensions
    quantity=2,                     # Learn exactly 2 macro-attractor states
    stored_pattern_as_static=False, # Trainable weights optimized via backprop
    state_pattern_as_static=True
)

classification_head = nn.Linear(total_input_size, 2)

optimizer = optim.Adam(
    list(chnn_classifier.parameters()) + list(classification_head.parameters()), 
    lr=0.002
)
criterion = nn.BCEWithLogitsLoss()



In [3]:
# ==========================================
# 3. THE BACKPROPAGATION LEARNING LOOP
# ==========================================
print("\nInitiating Backpropagation Loop on Full Trajectories...")
for epoch in range(101):
    optimizer.zero_grad()
    
    retrieved_features = chnn_classifier(X_train).squeeze(1)
    predictions = classification_head(retrieved_features)
    
    loss = criterion(predictions, Y_train)
    loss.backward()
    optimizer.step()
    
    if epoch % 25 == 0:
        print(f" Epoch {epoch:03d} | Backprop Classification Loss: {loss.item():.5f}")

print("\nHopfield Training Complete. Full Attractor Spaces Locked.")




Initiating Backpropagation Loop on Full Trajectories...
 Epoch 000 | Backprop Classification Loss: 0.76313
 Epoch 025 | Backprop Classification Loss: 41.87645
 Epoch 050 | Backprop Classification Loss: 96.58533
 Epoch 075 | Backprop Classification Loss: 29.95547
 Epoch 100 | Backprop Classification Loss: 32.53208

Hopfield Training Complete. Full Attractor Spaces Locked.


In [4]:
# ==========================================
# 4. TESTING ON UNSEEN DATA WITH 20-FRAME CONSTRAINT
# ==========================================
test_early_frames = 20  
print(f"\nEvaluating unseen test trajectories (runs 35-49) hidden after frame {test_early_frames}...")
print("-" * 80)
print(f"{'FILE NAME':<30} | {'ACTUAL MODE':<15} | {'PREDICTED MODE':<15} | {'STATUS':<10}")
print("-" * 80)

chnn_classifier.eval()
classification_head.eval()

total_tests = 0
correct_predictions = 0

with torch.no_grad():
    for run_id in range(35, 50):
        for true_mode in ['killing', 'non-killing']:
            filename = f"data_{true_mode}_{run_id}.npy"
            test_path = os.path.join(data_dir, filename)
            test_data = np.load(test_path).copy()
            
            # STRATEGIC MASKING: Wipe out all future frames (20 to 120) with zeros
            test_data[:, test_early_frames:, :] = 0.0
            
            test_masked_flat = test_data.transpose(1, 0, 2).flatten()
            test_query = torch.tensor(test_masked_flat, dtype=torch.float32).unsqueeze(0).unsqueeze(0) # [1, 1, 12000]
            
            # Run inference through the fully trained landscape
            output_features = chnn_classifier(test_query).squeeze(1)
            logits = classification_head(output_features)
            
            predicted_idx = torch.argmax(logits, dim=1).item()
            actual_idx = 0 if true_mode == 'killing' else 1
            
            predicted_mode = 'killing' if predicted_idx == 0 else 'non-killing'
            
            status = "✓ MATCH" if predicted_mode == true_mode else "✗ MISMATCH"
            print(f"{filename:<30} | {true_mode:<15} | {predicted_mode:<15} | {status:<10}")
            
            total_tests += 1
            if predicted_idx == actual_idx:
                correct_predictions += 1




Evaluating unseen test trajectories (runs 35-49) hidden after frame 20...
--------------------------------------------------------------------------------
FILE NAME                      | ACTUAL MODE     | PREDICTED MODE  | STATUS    
--------------------------------------------------------------------------------
data_killing_35.npy            | killing         | non-killing     | ✗ MISMATCH
data_non-killing_35.npy        | non-killing     | non-killing     | ✓ MATCH   
data_killing_36.npy            | killing         | non-killing     | ✗ MISMATCH
data_non-killing_36.npy        | non-killing     | non-killing     | ✓ MATCH   
data_killing_37.npy            | killing         | non-killing     | ✗ MISMATCH
data_non-killing_37.npy        | non-killing     | non-killing     | ✓ MATCH   
data_killing_38.npy            | killing         | non-killing     | ✗ MISMATCH
data_non-killing_38.npy        | non-killing     | non-killing     | ✓ MATCH   
data_killing_39.npy            | killing   

In [ ]:
    # ==========================================
    # 5. FINAL RESULTS REPORT
    # ==========================================
    accuracy = (correct_predictions / total_tests) * 100
    print("-" * 80)
    print("\n================ SYSTEM CLASSIFICATION REPORT ================")
    print(f"Total Unseen Simulations Evaluated: {total_tests}")
    print(f"Testing Constraints: Frames {test_early_frames}-120 completely masked out (Zeroed)")
    print(f"Hopfield Predictive Classification Accuracy: {accuracy:.2f}%")

--------------------------------------------------------------------------------

================ SYSTEM CLASSIFICATION REPORT ================
Total Unseen Simulations Evaluated: 30
Testing Constraints: Frames 20-120 completely masked out (Zeroed)
Hopfield Predictive Classification Accuracy: 50.00%
